<a href="https://colab.research.google.com/github/Akash9888/thesis_code/blob/master/ner_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import spacy
import torch
import os
from tqdm import tqdm
from google.colab import drive

drive.mount('/content/drive')


is_using_gpu = spacy.prefer_gpu()
print(f"--- GPU ACCELERATION ACTIVE: {is_using_gpu} ---")

nlp = spacy.load("en_core_web_sm", disable=["tagger", "parser", "attribute_ruler", "lemmatizer"])

INPUT_FILE = '/content/drive/MyDrive/ThesisData/news_data_small_phase1.csv'
OUTPUT_FILE = '/content/drive/MyDrive/ThesisData/news_with_entities.csv'

def run_ner_only():
    if not os.path.exists(INPUT_FILE):
        print("Error: Input file not found!")
        return


    df = pd.read_csv(INPUT_FILE)
    print(f"Loaded {len(df)} rows. Starting NER...")

    headlines = df['headline'].astype(str).tolist()
    extracted_entities = []


    print("Extracting ORG and PERSON entities...")
    for doc in tqdm(nlp.pipe(headlines, batch_size=512), total=len(headlines)):
        ents = [f"{e.text}({e.label_})" for e in doc.ents if e.label_ in ['ORG', 'PERSON']]
        extracted_entities.append(", ".join(ents) if ents else "")


    df['entities'] = extracted_entities


    print("\n[CHECK] NER Results (Top 5):")
    print(df[['headline', 'entities']].head(5))
    print("-" * 30)


    df.to_csv(OUTPUT_FILE, index=False)
    print(f"SUCCESS: {OUTPUT_FILE} saved with extracted entities.")

if __name__ == "__main__":
    run_ner_only()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
--- GPU ACCELERATION ACTIVE: False ---
Loaded 199404 rows. Starting NER...
Extracting ORG and PERSON entities...


100%|██████████| 199404/199404 [02:43<00:00, 1216.47it/s]



[CHECK] NER Results (Top 5):
                                            headline  \
0  The government official in charge of ethics ju...   
1  Republicans totally outsmarted the mainstream ...   
2  How the Clinton campaign is making #ThatMexica...   
3  In Turkey, Music Takes You Where a Travel Visa...   
4                            Peter Thiel vs. the FDA   

                        entities  
0                     Trump(ORG)  
1                                 
2                Clinton(PERSON)  
3                                 
4  Peter Thiel(PERSON), FDA(ORG)  
------------------------------
SUCCESS: /content/drive/MyDrive/ThesisData/news_with_entities.csv saved with extracted entities.
